# Mangrove Pulse: repaired-runner reproduction

This check runs candidate `04fa7c18ccc39e6cf5656686b39b53ca82a81ce5` from an empty working directory. It reuses the team's cached input archive, fetches only changed public repository files, verifies all candidate payload hashes, installs the baseline requirements, runs software tests and reproduces the frozen baseline and disagreement audit.

The results establish computational consistency of this candidate—not independent cover accuracy, aircraft performance or ecological benefit. [Repository](https://github.com/Salim-Ammar/mangrove-pulse) · [Earlier reproduction](https://github.com/Salim-Ammar/mangrove-pulse/blob/main/docs/REVIEW_CHECKS_20261012.md)


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import hashlib, io, json, os, subprocess, sys, tempfile, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

COMMIT = '04fa7c18ccc39e6cf5656686b39b53ca82a81ce5'
MANIFEST_HASH = 'e4bf8ad69c77c94467b13ffeca1d807fe8d7ff72eb404ae68fbc74a21fe17cbf'
SNAPSHOT_HASH = '28aa20c925b8ddccf327f30e5e1408e716aa397960e7e7b7e1da7b2ddff6ccc6'
BASE_URL = 'https://raw.githubusercontent.com/Salim-Ammar/mangrove-pulse/' + COMMIT + '/'
work = Path(tempfile.mkdtemp(prefix='mp_engineering_'))
root = work / 'candidate'; root.mkdir()
results = work / 'results'; results.mkdir()
record = {'candidate': COMMIT, 'started_utc': datetime.now(timezone.utc).isoformat(),
          'status': 'running', 'steps': [], 'scope': 'cached-input baseline/audit reproduction and software tests'}
def save_record():
    (results / 'execution_record.json').write_text(json.dumps(record, indent=2) + '\n')
def step(name, command, env=None):
    started = time.perf_counter()
    with (results / (name + '.log')).open('w') as log:
        proc = subprocess.run(command, cwd=root, env=env, stdout=log,
                              stderr=subprocess.STDOUT, timeout=600)
    record['steps'].append({'name': name, 'returncode': proc.returncode,
                            'seconds': time.perf_counter() - started})
    save_record()
    if proc.returncode:
        raise RuntimeError(name + ' failed; see its log')
try:
    manifest_bytes = urllib.request.urlopen(BASE_URL + 'SHA256SUMS.json', timeout=60).read()
    if hashlib.sha256(manifest_bytes).hexdigest() != MANIFEST_HASH:
        raise ValueError('Candidate manifest hash mismatch')
    manifest = json.loads(manifest_bytes)
    snapshot = Path('/content/drive/MyDrive/Mangrove_Pulse_Release_cb4b400.zip').read_bytes()
    if hashlib.sha256(snapshot).hexdigest() != SNAPSHOT_HASH:
        raise ValueError('Cached snapshot hash mismatch')
    reused = downloaded = downloaded_bytes = 0
    with zipfile.ZipFile(io.BytesIO(snapshot)) as old:
        old_manifest = json.loads(old.read('submission/SHA256SUMS.json'))
        for name, expected in manifest.items():
            path = PurePosixPath(name)
            if path.is_absolute() or '..' in path.parts or path.as_posix() != name or ':' in name or chr(92) in name:
                raise ValueError('Unsafe candidate path')
            if old_manifest.get(name) == expected:
                payload = old.read('submission/' + name); reused += 1
            else:
                payload = urllib.request.urlopen(BASE_URL + name, timeout=60).read()
                downloaded += 1; downloaded_bytes += len(payload)
            if hashlib.sha256(payload).hexdigest() != expected:
                raise ValueError('Payload hash mismatch: ' + name)
            target = root / name; target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(payload)
    (root / 'SHA256SUMS.json').write_bytes(manifest_bytes)
    record.update(manifest_sha256=MANIFEST_HASH, payloads=len(manifest),
                  reused_payloads=reused, changed_payloads_downloaded=downloaded,
                  changed_bytes_downloaded=downloaded_bytes, cached_snapshot_sha256=SNAPSHOT_HASH)
    save_record()
    venv = work / 'venv'
    step('create_environment', [sys.executable, '-m', 'venv', str(venv)])
    python = str(venv / 'bin/python')
    step('install', [python, '-m', 'pip', 'install', '-r', str(root / 'requirements.txt')])
    step('packages', [python, '-m', 'pip', 'freeze'])
    env = dict(os.environ, MPLBACKEND='Agg', PYTHONDONTWRITEBYTECODE='1')
    record['kernel_python'] = sys.version
    step('software_tests', [python, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], env)
    step('reproduction', [python, str(root / 'src/run_submission.py'), '--root', str(root),
                         '--out', str(results / 'reproduction')], env)
    status = json.loads((results / 'reproduction/run_status.json').read_text())
    if status['status'] != 'succeeded':
        raise ValueError('Reproduction did not succeed')
    record['status'] = 'succeeded'
except Exception as exc:
    record.update(status='failed', error=str(exc))
    raise
finally:
    record['finished_utc'] = datetime.now(timezone.utc).isoformat()
    save_record()
    print(json.dumps(record, indent=2))


{
  "candidate": "04fa7c18ccc39e6cf5656686b39b53ca82a81ce5",
  "started_utc": "2026-10-11T23:23:49.113544+00:00",
  "status": "failed",
  "steps": [
    {
      "name": "create_environment",
      "returncode": 1,
      "seconds": 0.11602592500003084
    }
  ],
  "scope": "cached-input baseline/audit reproduction and software tests",
  "manifest_sha256": "e4bf8ad69c77c94467b13ffeca1d807fe8d7ff72eb404ae68fbc74a21fe17cbf",
  "payloads": 170,
  "reused_payloads": 125,
  "changed_payloads_downloaded": 45,
  "changed_bytes_downloaded": 13661621,
  "cached_snapshot_sha256": "28aa20c925b8ddccf327f30e5e1408e716aa397960e7e7b7e1da7b2ddff6ccc6",
  "error": "create_environment failed; see its log",
  "finished_utc": "2026-10-11T23:24:08.072791+00:00"
}


RuntimeError: create_environment failed; see its log

In [5]:
print((results / 'create_environment.log').read_text())

Error: Command '['/tmp/mp_engineering_n0wswuhb/venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.



In [7]:
exec("import hashlib, json, os, shutil, subprocess, sys, tempfile, time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\ncache = Path('/tmp/mp_engineering_n0wswuhb')\nroot = cache / 'candidate'\nassert hashlib.sha256((root / 'SHA256SUMS.json').read_bytes()).hexdigest() == 'e4bf8ad69c77c94467b13ffeca1d807fe8d7ff72eb404ae68fbc74a21fe17cbf'\nmanifest = json.loads((root / 'SHA256SUMS.json').read_text())\nfor name, expected in manifest.items():\n    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name\nwork = Path(tempfile.mkdtemp(prefix='mp_engineering_retry_'))\nresults = work / 'results'; results.mkdir()\nshutil.copyfile(cache / 'results/execution_record.json', results / 'initial_failed_record.json')\nshutil.copyfile(cache / 'results/create_environment.log', results / 'initial_create_environment.log')\nrecord = {'candidate':'04fa7c18ccc39e6cf5656686b39b53ca82a81ce5', 'status':'running',\n          'started_utc':datetime.now(timezone.utc).isoformat(), 'steps':[],\n          'payloads':len(manifest), 'kernel_python':sys.version,\n          'scope':'affected setup/reproduction retry; verified existing candidate cache, fresh environment and outputs'}\ndef save_record():\n    (results / 'execution_record.json').write_text(json.dumps(record, indent=2)+'\\n')\ndef step(name, command, env=None):\n    started=time.perf_counter()\n    with (results / (name+'.log')).open('w') as log:\n        proc=subprocess.run(command,cwd=root,env=env,stdout=log,stderr=subprocess.STDOUT,timeout=600)\n    record['steps'].append({'name':name,'returncode':proc.returncode,'seconds':time.perf_counter()-started})\n    save_record()\n    if proc.returncode: raise RuntimeError(name+' failed; see log')\ntry:\n    venv=work / 'venv'\n    step('create_environment',[sys.executable,'-m','venv','--without-pip',str(venv)])\n    python=str(venv / 'bin/python')\n    step('install',[sys.executable,'-m','pip','--python',python,'install','-r',str(root / 'requirements.txt')])\n    step('packages',[sys.executable,'-m','pip','--python',python,'freeze','--all'])\n    env=dict(os.environ,MPLBACKEND='Agg',PYTHONDONTWRITEBYTECODE='1')\n    step('software_tests',[python,'-m','unittest','discover','-s','tests','-v'],env)\n    step('reproduction',[python,str(root / 'src/run_submission.py'),'--root',str(root),'--out',str(results / 'reproduction')],env)\n    assert json.loads((results / 'reproduction/run_status.json').read_text())['status']=='succeeded'\n    record['status']='succeeded'\nexcept Exception as exc:\n    record.update(status='failed',error=str(exc))\n    raise\nfinally:\n    record['finished_utc']=datetime.now(timezone.utc).isoformat()\n    save_record()\n    print(json.dumps(record,indent=2))\n")

{
  "candidate": "04fa7c18ccc39e6cf5656686b39b53ca82a81ce5",
  "status": "succeeded",
  "started_utc": "2026-10-11T23:29:45.912644+00:00",
  "steps": [
    {
      "name": "create_environment",
      "returncode": 0,
      "seconds": 0.16981145999989167
    },
    {
      "name": "install",
      "returncode": 0,
      "seconds": 22.495558311999957
    },
    {
      "name": "packages",
      "returncode": 0,
      "seconds": 0.7665104889999839
    },
    {
      "name": "software_tests",
      "returncode": 0,
      "seconds": 0.7167835650000143
    },
    {
      "name": "reproduction",
      "returncode": 0,
      "seconds": 7.046362663000082
    }
  ],
  "payloads": 170,
  "kernel_python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "scope": "affected setup/reproduction retry; verified existing candidate cache, fresh environment and outputs",
  "finished_utc": "2026-10-11T23:30:17.110142+00:00"
}


In [9]:
from google.colab import files
package = work / 'engineering_04fa7c1_results.zip'
with zipfile.ZipFile(package, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(results.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(results))
print('Status:', record['status'])
print('ZIP bytes:', package.stat().st_size)
print('SHA-256:', hashlib.sha256(package.read_bytes()).hexdigest())
files.download(str(package))


Status: succeeded
ZIP bytes: 1687436
SHA-256: 6ef393329fdd3564d85b5794ca6715ecc63e6a8aee93f336aa47c5e752519a04


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>